In [14]:
# =========================
# 1. 라이브러리 import
# =========================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

In [15]:
# =========================
# 2. 데이터 로드
# =========================
df = pd.read_csv('skin_irritation_2Ddesc.csv')

print("전체 데이터 shape:", df.shape)

전체 데이터 shape: (39, 220)


In [16]:
# =========================
# 3. X / y 분리 + 전처리
# =========================
X = df.drop(columns=['label'])
y = df['label']

# 숫자형 feature만 사용
X = X.select_dtypes(include=[np.number])

# NaN, inf 처리
X = X.fillna(0)
X = X.replace([np.inf, -np.inf], 0)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (39, 217)
y shape: (39,)


In [17]:
# =========================
# 4. train / test split (🔥 먼저 해야 함)
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (31, 217)
Test size: (8, 217)


In [18]:
# =========================
# 5. VarianceThreshold (train 기준)
# =========================
selector_var = VarianceThreshold(threshold=0.01)

X_train_var = selector_var.fit_transform(X_train)
X_test_var = selector_var.transform(X_test)

print("After VarianceThreshold:", X_train_var.shape)

After VarianceThreshold: (31, 143)


In [64]:
# =========================
# 6. SelectKBest (train 기준)
# =========================
k = 60

selector_k = SelectKBest(score_func=f_classif, k=k)

X_train_sel = selector_k.fit_transform(X_train_var, y_train)
X_test_sel = selector_k.transform(X_test_var)

print("After SelectKBest:", X_train_sel.shape)

After SelectKBest: (31, 60)


In [65]:
# =========================
# 7. Scaling (train 기준)
# =========================
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled = scaler.transform(X_test_sel)

In [66]:
# =========================
# 8. 기본 MLP 모델
# =========================
model = MLPClassifier(
    hidden_layer_sizes=(10,),
    max_iter=2000,
    early_stopping=True,
    random_state=42
)

model.fit(X_train_scaled, y_train)

print("Train Accuracy:", model.score(X_train_scaled, y_train))
print("Test Accuracy:", model.score(X_test_scaled, y_test))

Train Accuracy: 0.5161290322580645
Test Accuracy: 0.125


In [73]:
# =========================
# 9. 하이퍼파라미터 실험
# =========================
results = []

hidden_options = [(50,), (100,), (50, 50)]
alpha_options = [0.0001, 0.001, 0.01]
lr_options = [0.001, 0.01]

k_values = [3, 4, 5]

for k in k_values:
    
    # feature 선택
    selector_k = SelectKBest(score_func=f_classif, k=k)
    X_train_sel = selector_k.fit_transform(X_train_var, y_train)
    X_test_sel = selector_k.transform(X_test_var)
    
    # scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_sel)
    X_test_scaled = scaler.transform(X_test_sel)
    
    # 🔥 hidden node ≤ feature 개수
    hidden_options = [
        (k,),
        (k-1,),
        (k, k-1),
        (max(1,k-1), max(1,k-2))
    ]
    
    for h in hidden_options:
        if h[0] <= k:  # 안전 체크
            
            model = MLPClassifier(
                hidden_layer_sizes=h,
                max_iter=2000,
                early_stopping=True,
                random_state=42
            )
            
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)
            
            acc = accuracy_score(y_test, y_pred)
            
            results.append({
                'k': k,
                'hidden_layer': h,
                'accuracy': acc
            })

# 결과 정리
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='accuracy', ascending=False)

print(results_df)


    k hidden_layer  accuracy
0   3         (3,)     0.750
1   3         (2,)     0.750
2   3       (3, 2)     0.750
4   4         (4,)     0.750
10  5       (5, 4)     0.750
5   4         (3,)     0.750
11  5       (4, 3)     0.750
6   4       (4, 3)     0.625
9   5         (4,)     0.500
8   5         (5,)     0.375
3   3       (2, 1)     0.250
7   4       (3, 2)     0.250


In [74]:
# =========================
# 10. 결과 정리
# =========================
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='accuracy', ascending=False)

print("\n=== Hyperparameter Results ===")
print(results_df)


=== Hyperparameter Results ===
    k hidden_layer  accuracy
0   3         (3,)     0.750
1   3         (2,)     0.750
2   3       (3, 2)     0.750
4   4         (4,)     0.750
10  5       (5, 4)     0.750
5   4         (3,)     0.750
11  5       (4, 3)     0.750
6   4       (4, 3)     0.625
9   5         (4,)     0.500
8   5         (5,)     0.375
3   3       (2, 1)     0.250
7   4       (3, 2)     0.250


In [75]:
# =========================
# 11. Best 모델 출력
# =========================
best = results_df.iloc[0]

print("\n=== Best Setting ===")
print(best)


=== Best Setting ===
k                  3
hidden_layer    (3,)
accuracy        0.75
Name: 0, dtype: object
